# ✈️ AeroLex — Building a Production-Grade RAG Pipeline

A hands-on build of a multi-stage Retrieval-Augmented Generation system over
**airline passenger-rights law** (EU 261/2004, US DOT 14 CFR 260, Delta Contracts
of Carriage).

**The pipeline:**
```
question
  └─▶ condense follow-up            (conversational memory)       [#8]
  └─▶ BM25 lexical  +  Gemini dense  (hybrid retrieval)           [#2,#3,#4]
  └─▶ reciprocal rank fusion         (merge the two)              [#5]
  └─▶ cross-encoder re-ranking       (precise top-k)              [#6]
  └─▶ grounded answer + [[n]] cites  (+ long-term profile)        [#7,#9]
```

**How to run:** pick the **`Python (aerolex RAG)`** kernel, then run top to bottom.
Everything works offline (extractive answers); with a `GEMINI_API_KEY` you get full
generation. You built each stage? The final section assembles them with the
`aerolex` library.


## 0 · Setup

In [ ]:
import os

import numpy as np
from aerolex import config
from aerolex.loaders import load_documents
from aerolex.chunking import chunk_corpus, load_chunks
from aerolex.store import InMemoryVectorStore

key = config.load_api_key()
print("GEMINI_API_KEY:", "✓ found" if key else "✗ not found — offline fallback mode")
print("gen model:", config.GEN_MODEL, "| embed model:", config.EMBED_MODEL, "| dim:", config.EMBED_DIM)

GEMINI_API_KEY: ✓ found
gen model: gemini-flash-latest | embed model: gemini-embedding-001 | dim: 768


## 1 · Hierarchical Chunking  ·  Feature #1

Legal text is a **hierarchy** (Article / § / RULE → paragraph). We split on those
*logical* boundaries — never fixed-size windows — and attach citation metadata, so
an answer can cite *"EU 261/2004 — Article 7"* instead of *"chunk 42"*.

In [4]:
docs = load_documents()
for d in docs:
    print(f"{d.doc_id:16s} {d.doc_type:6s} {len(d.text):>7,d} chars  {d.title}")

# Prefer the pre-built cache (instant); otherwise chunk live.
try:
    chunks = load_chunks()
    print(f"\nloaded {len(chunks)} chunks from cache")
except FileNotFoundError:
    chunks = chunk_corpus(docs)
    print(f"\nchunked live: {len(chunks)} chunks")

by_id = {c.chunk_id: c for c in chunks}
art7 = next(c for c in chunks if c.section_label == "Article 7")
print("\nExample chunk —", art7.citation)
print(art7.text[:320], "…")

eu261            eu      27,544 chars  EU Regulation (EC) No 261/2004
cfr260           cfr     26,287 chars  US DOT 14 CFR Part 260
delta_domestic   delta   78,663 chars  Delta Domestic General Rules Tariff (Contract of Carriage)
delta_intl       delta  118,099 chars  Delta International General Rules Tariff (Contract of Carriage)

chunked live: 234 chunks

Example chunk — EU Regulation (EC) No 261/2004 — Article 7
1.
Where reference is made to this Article, passengers shall
receive compensation amounting to:
(a) EUR 250 for all flights of 1 500 kilometres or less;
(b) EUR 400 for all intra-Community flights of more than
1 500 kilometres, and for all other flights between 1 500
and 3 500 kilometres;
(c) EUR 600 for all flights no …


## 2 · Lexical Search with BM25  ·  Feature #2

Dense search blurs exact tokens, but statute codes must match **exactly**
(`§ 260.6`, `261/2004`). BM25 is a sparse bag-of-words scorer — the trick is a
**tokenizer that preserves those references** as single tokens.

### 🛠️ TODO: write `tokenize`

In [1]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Keep words and dotted/slashed numeric refs as single tokens.
_TOKEN = re.compile(r"[a-z]+|\d+(?:[./-]\d+)*")


def tokenize(text: str) -> list[str]:
    # TODO Fetch list of keywords
    tokens = re.findall(_TOKEN, text.lower())
    return [token for token in tokens if token not in ENGLISH_STOP_WORDS]

assert tokenize("What is the refund for a cancelled flight?") == ["refund", "cancelled", "flight"]
print("tokenize OK")

tokenize OK


In [6]:
from aerolex.bm25 import BM25Index

bm25 = BM25Index([c.chunk_id for c in chunks], [c.searchable_text for c in chunks])
for cid, score in bm25.search("§ 260.6 refund for a cancelled flight", top_k=3):
    print(f"{score:6.2f}  {by_id[cid].citation}")

 16.02  US DOT 14 CFR Part 260 — § 260.6 (part 6/6)
 15.18  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)
 15.00  US DOT 14 CFR Part 260 — § 260.6 (part 3/6)


## 3 · Dense Semantic Search + In-Memory Store  ·  Features #3 & #4

Gemini embeds every chunk into a vector; the whole corpus is one NumPy matrix.
Similarity is just a **normalized dot product** (cosine) — no vector database.


### 🛠️ TODO: write `cosine_search` (normalized dot product top-k)

In [ ]:
def cosine_search(query_vec, matrix, ids, top_k=5):

    # TODO: 1) L2-normalize query_vec and each row of matrix
    q_hat = query_vec / np.sqrt(np.sum(query_vec ** 2))
    matrix_hat = matrix / np.sqrt(np.sum(matrix ** 2, axis=1, keepdims=True))
    #       2) scores = matrix @ query   (cosine, since normalized)
    scores = matrix_hat @ q_hat
    #       3) return the top_k (id, score) pairs, highest first
    top_k = np.argsort(-scores)[:top_k]
    top_chunks = [(ids[i], scores[i]) for i in top_k]
    return top_chunks

# sanity: a row is most similar to itself (cosine ≈ 1.0)
_hit = cosine_search(store.matrix[0], store.matrix, store.ids, top_k=1)[0]
print(_hit[0] == store.ids[0], round(_hit[1], 4))

True 1.0


In [ ]:
# Embed a real query (1 API call) and search. Needs a key; else we reuse a chunk vector.
query = "What is the refund for a cancelled flight"

query_vector = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM).embed_query(query)


dense_hits = cosine_search(query_vector, store.matrix, store.ids, top_k=5)
for cid, s in dense_hits:
    print(f"{s:5.3f}  {by_id[cid].citation}")

0.726  EU Regulation (EC) No 261/2004 — Article 7
0.684  EU Regulation (EC) No 261/2004 — Article 6
0.657  US DOT 14 CFR Part 260 — § 260.2 (part 5/6)
0.654  EU Regulation (EC) No 261/2004 — Article 9
0.652  EU Regulation (EC) No 261/2004 — Article 14


## 4 · Reciprocal Rank Fusion  ·  Feature #5

BM25 scores and cosine scores live on different scales, so we fuse by **rank**:
each list contributes `1 / (k + rank)` to an id's score (`k = 60`). Documents that
rank well in *either* retriever bubble up.

### 🛠️ TODO: write `rrf`

In [ ]:
def rrf(rankings, k=60, top_k=10):
    scores = {}
    for ranked in rankings:
        for rank, cid in enumerate(ranked):
            chunk_score = scores.get(cid, 0.0)
            # TODO for each chunk compute the RRF Score
            scores[cid] = chunk_score + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:top_k]

assert rrf([["b", "a", "c"], ["b", "a", "d"]])[0][0] == "b"
print("rrf OK")

rrf OK


In [12]:
bm25_ids = [cid for cid, _ in bm25.search(query, top_k=20)]
dense_ids = [cid for cid, _ in dense_hits]
fused = rrf([bm25_ids, dense_ids], k=config.RRF_K, top_k=10)
print("Fused top-5:")
for cid, s in fused[:5]:
    print(f"{s:.4f}  {by_id[cid].citation}")

Fused top-5:
0.0297  EU Regulation (EC) No 261/2004 — Article 7
0.0286  EU Regulation (EC) No 261/2004 — Article 6
0.0284  EU Regulation (EC) No 261/2004 — Article 14
0.0164  EU Regulation (EC) No 261/2004 — Preamble (part 5/6)
0.0161  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)


## 5 · Cross-Encoder Re-Ranking  ·  Feature #6

First-stage retrieval scores query and document *independently*. A cross-encoder
reads them **together** (deep cross-attention) and is far more precise — so we run
it only over the small fused pool. (Model is pre-cached; runs locally.)

In [13]:
from aerolex.rerank import CrossEncoderReranker

candidates = [(cid, by_id[cid].searchable_text) for cid, _ in fused]
reranked = CrossEncoderReranker().rerank(query, candidates, top_k=config.RERANK_TOP_K)
print("Re-ranked top chunks:")
for cid, s in reranked:
    print(f"{s:7.2f}  {by_id[cid].citation}")
final_chunks = [by_id[cid] for cid, _ in reranked]

/opt/anaconda3/envs/rag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1021 23:29:06.785000 52409 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 19469.58it/s]


Re-ranked top chunks:
   3.03  EU Regulation (EC) No 261/2004 — Article 7
  -0.90  EU Regulation (EC) No 261/2004 — Preamble (part 1/6)
  -1.79  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)
  -2.23  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
  -2.83  US DOT 14 CFR Part 260 — § 260.6 (part 3/6)


## 6 · Grounded Answer with Inline Citations  ·  Feature #7

The model answers **only** from numbered passages and tags each claim with `[[n]]`.
A regex post-processor then validates those tags against the real passages and
**drops hallucinated ones** — the difference between "looks cited" and "is cited".

### 🛠️ TODO: write `resolve_citations`

In [14]:
CITE = re.compile(r"\[\[(\d+)\]\]")

def resolve_citations(answer, index):
    used = []
    def keep(m):
        n = int(m.group(1))
        if n in index:
            if n not in used:
                used.append(n)
            return m.group(0)
        return ""                      # drop a citation to a non-existent passage
    cleaned = CITE.sub(keep, answer)
    return cleaned, [f"[[{n}]] {index[n]}" for n in sorted(used)]

_c, _r = resolve_citations("A [[1]] and B [[9]].", {1: "EU261 Art 7", 2: "EU261 Art 5"})
print(_c, "|", _r)

A [[1]] and B . | ['[[1]] EU261 Art 7']


In [15]:
from aerolex.generate import Generator

result = Generator().generate(query, final_chunks)
print("OFFLINE" if result["offline"] else "GEMINI", "answer:\n")
print(result["answer"])
print("\nReferences:")
for r in result["references"]:
    print("  ", r)

GEMINI answer:

The provided context does not state how much compensation is owed specifically for a long delayed flight [[1], [2]]. 

While Regulation (EC) No 261/2004 notes that it establishes common rules on compensation and assistance to passengers in the event of denied boarding and of cancellation or long delay of flights [[2]], the text of Article 7 only specifies amounts where reference is made to that Article, mentioning only denial of boarding or cancellation [[1]]. Under Article 7, the established compensation amounts are:
* EUR 250 for all flights of 1 500 kilometres or less [[1]].
* EUR 400 for all intra-Community flights of more than 1 500 kilometres, and for all other flights between 1 500 and 3 500 kilometres [[1]].
* EUR 600 for all flights not falling under the first two categories [[1]].

These amounts may be reduced by 50 % if alternative re-routing arrives within two, three, or four hours of the scheduled arrival time depending on flight distance, but the provided 

## 7 · Conversational Memory  ·  Feature #8

Follow-ups like *"what about in the US?"* are meaningless to a retriever alone. We
**condense** the chat history + follow-up into a standalone query before searching.

In [16]:
from aerolex.memory import ConversationMemory

mem = ConversationMemory()
mem.add_user(query)
mem.add_assistant(result["answer"][:160])
print("follow-up : what about in the US?")
print("standalone:", mem.condense("what about in the US?"))

follow-up : what about in the US?
standalone: How much compensation for a long delayed flight in the US?


## 8 · Long-Term Persistence  ·  Feature #9

Facts a user states persist across sessions in SQLite and can be injected into
future prompts.

In [17]:
from aerolex.persistence import ProfileStore

prof = ProfileStore()
prof.remember("demo_user", "home_airport", "JFK")
prof.remember("demo_user", "status", "Delta Platinum Medallion")
print(prof.context_block("demo_user"))

Known facts about the user:
- home_airport: JFK
- status: Delta Platinum Medallion


## 9 · Adaptive Feedback — FLAIR  ·  Feature #10

We log `(query, chunk, 👍/👎)` triplets and nudge future rankings by **query-weighted
votes**: a past vote counts only when the new query is similar to the one it was
cast for — so feedback generalizes to unseen-but-similar questions.

In [18]:
from aerolex.feedback import FeedbackStore

fb = FeedbackStore()
top_id = reranked[0][0]
fb.record(query, qvec, top_id, +1)                 # 👍 the best chunk
ranked = [(cid, s) for cid, s in reranked]
adjusted = fb.apply(ranked, qvec)
print("before:", [by_id[c].section_label for c, _ in ranked])
print("after :", [by_id[c].section_label for c, _ in adjusted])

before: ['Article 7', 'Preamble', '§ 260.6', '§ 260.6', '§ 260.6']
after : ['Article 7', 'Preamble', '§ 260.6', '§ 260.6', '§ 260.6']


## 10 · The Full Pipeline

Now assemble every stage with the `aerolex` library — the productionized version of
what you just built — and ask a question end to end.

In [19]:
from aerolex.pipeline import RagPipeline

if key:
    from aerolex.embeddings import GeminiEmbedder
    pipe = RagPipeline(
        chunks=chunks,
        store=store,
        bm25=bm25,
        embedder=GeminiEmbedder(),
        reranker=CrossEncoderReranker(),
        generator=Generator(),
    )
    out = pipe.answer("If my 4000 km flight is cancelled, what am I owed?",
                      memory=mem, profile=prof, user_id="demo_user")
    print(out["answer"])
    print("\nReferences:")
    for r in out["references"]:
        print("  ", r)
else:
    print("Set GEMINI_API_KEY to run the full generative pipeline.")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9937.31it/s]


What you are owed depends on the applicable regulations and airline terms governing your flight:

**Under EU Regulation (EC) No 261/2004 (if applicable):**
* Because a 4,000 km flight exceeds 3,500 km, it falls under Article 7(1)(c), which provides compensation of EUR 600 [[1]].
* The operating air carrier may reduce this compensation by 50% if you are offered re-routing to your final destination on an alternative flight that arrives no more than four hours after your original scheduled arrival time [[1]].
* Compensation is payable in cash, by electronic bank transfer, bank orders, bank cheques, or, with your signed agreement, in travel vouchers and/or other services [[1]].
* In addition to compensation, you must be offered assistance in accordance with Article 8 and Article 9 [[5]].
* Compensation is not required if you were informed of the cancellation at least two weeks prior to departure, or if you were given notice between two weeks and seven days prior with re-routing departing n

## 11 · Where to go next — citations, state of the art

What we built is **Level 1** citation: the model self-tags `[[n]]`, and we validate
the *number is real*. State-of-the-art adds **verification**:

- **Level 2** structured output — model returns `{sentence, support_ids, quote}`.
- **Level 3** quote-grounded spans — match the quote back into the source for exact
  offsets; drop unverifiable quotes.
- **Level 4** NLI / entailment (ALCE) — check the passage actually *entails* the
  claim; score citation **precision & recall**.

The retrieval stack (chunking → hybrid → fusion → re-rank) is where most answer
quality is won. 🎓 You built all of it.